# LAB 04 — Mục 23: Application (Option B — Topic Classification)\nNotebook Colab phân loại chủ đề với AG News: World/Politics, Sports, Business, Technology. Dùng TF-IDF + Logistic Regression, chia train/validation/test theo tỉ lệ 60/20/20, in metrics và confusion matrix.

In [ ]:
# LAB 04 — Mục 23: Application (Option B — Topic Classification)

# Colab: cài đặt/thư viện, tải AG News, chia train/validation/test 60/20/20,
# huấn luyện TF-IDF + Logistic Regression và báo cáo metrics.

# 1. Cài thư viện (chạy một lần trên Colab)
!pip -q install datasets scikit-learn pandas

# 2. Nạp AG News
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# AG News gốc có 4 lớp tương ứng Business, Sci/Tech, Sports, World.
# Chọn subset cân bằng để chạy nhanh; có thể tăng N_PER_CLASS hoặc dùng toàn bộ dữ liệu.
LABELS = ["World/Politics", "Sports", "Business", "Technology"]
N_PER_CLASS = 5000
SEED = 42

ds = load_dataset("fancyzhx/ag_news")
df = pd.DataFrame(ds["train"]).rename(columns={"text": "text", "label": "label"})
df["label"] = df["label"].astype(int)
df["text"] = df["text"].fillna("").astype(str)
df = df.groupby("label", group_keys=False).sample(
    n=min(N_PER_CLASS, df.groupby("label").size().min()), random_state=SEED
).reset_index(drop=True)
df["topic"] = df["label"].map(dict(enumerate(LABELS)))

print("Số mẫu:", len(df))
print("\nPhân bố nhãn:")
print(df["topic"].value_counts().to_string())

# 3. Chia phân tầng 60/20/20; giữ test độc lập đến bước đánh giá cuối
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df["text"], df["label"], test_size=0.20, random_state=SEED, stratify=df["label"]
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEED, stratify=y_trainval
)
print(f"\nTrain={len(X_train):,}; Validation={len(X_val):,}; Test={len(X_test):,}")

# 4. Pipeline: vectorizer chỉ fit trên train để tránh data leakage
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000,
                              strip_accents="unicode", sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED))
])
model.fit(X_train, y_train)
print("Vocabulary size:", len(model.named_steps["tfidf"].vocabulary_))

# 5. Đánh giá validation để kiểm tra pipeline
val_pred = model.predict(X_val)
p, r, f1, _ = precision_recall_fscore_support(
    y_val, val_pred, average="macro", zero_division=0
)
print(f"\nValidation — Accuracy={accuracy_score(y_val, val_pred):.4f}, "
      f"Macro Precision={p:.4f}, Macro Recall={r:.4f}, Macro F1={f1:.4f}")

# 6. Đánh giá cuối cùng trên test
test_pred = model.predict(X_test)
p, r, f1, _ = precision_recall_fscore_support(
    y_test, test_pred, average="macro", zero_division=0
)
print(f"\nTest — Accuracy={accuracy_score(y_test, test_pred):.4f}, "
      f"Macro Precision={p:.4f}, Macro Recall={r:.4f}, Macro F1={f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, test_pred, labels=range(4),
                            target_names=LABELS, digits=4, zero_division=0))

cm = confusion_matrix(y_test, test_pred, labels=range(4))
print("Confusion matrix (hàng=nhãn thật, cột=dự đoán; thứ tự:", LABELS, ")")
print(cm)

# 7. Dự đoán thử trên văn bản mới
examples = [
    "The team won the championship after a dramatic final match.",
    "The central bank announced new interest rate decisions.",
    "The company unveiled a new AI processor for data centers.",
    "Parliament debated the government's proposed election reforms."
]
for text, pred in zip(examples, model.predict(examples)):
    print(f"\n{text}\n=> {LABELS[pred]}")

# Trả lời mục 23 — Option B:
# Đây là bài toán phân loại chủ đề 4 lớp. TF-IDF biểu diễn các từ/cụm từ,
# Logistic Regression dự đoán chủ đề. Macro-F1 đánh giá cân bằng từng lớp;
# xem confusion matrix để nhận ra các chủ đề hay bị nhầm (đặc biệt Business/Technology
# có thể dùng chung từ vựng). Kết quả thực tế được in ở trên khi chạy notebook.


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Số mẫu: 20000

Phân bố nhãn:
topic
World/Politics    5000
Sports            5000
Business          5000
Technology        5000

Train=12,000; Validation=4,000; Test=4,000
Vocabulary size: 64053

Validation — Accuracy=0.8980, Macro Precision=0.8978, Macro Recall=0.8980, Macro F1=0.8977

Test — Accuracy=0.8870, Macro Precision=0.8867, Macro Recall=0.8870, Macro F1=0.8865

Classification report:
                precision    recall  f1-score   support

World/Politics     0.9062    0.8700    0.8878      1000
        Sports     0.9213    0.9720    0.9460      1000
      Business     0.8643    0.8410    0.8525      1000
    Technology     0.8547    0.8650    0.8598      1000

      accuracy                         0.8870      4000
     macro avg     0.8867    0.8870    0.8865      4000
  weighted avg     0.8867    0.8870    0.8865      4000

Confusion matrix (hàng=nhãn thật, cột=dự đoán; thứ tự: ['World/Politics', 'Sports', 'Business', 'Technology'] )
[[870  43  54  33]
 [ 10 972   8  10]
 [ 